# 🧪 W6-D1 概念实验：LLM Agent 基本架构

> 配套阅读：`第6周-Day1-LLM-Agent基本架构.md`（三大组件、ReAct 流程、业务场景在那边）
>
> 本 notebook 用可运行的最小实现回答四个问题：
> 1. **Agent = LLM + 工具 + 记忆 + 循环**，一个能跑的 ReAct 循环长什么样？
> 2. **Agent vs 传统编程**：为什么规则引擎没法穷举，Agent 却能应付没见过的组合请求？
> 3. **记忆组件**：全量保留 / 滑动窗口 / 窗口+摘要，token 成本差多少？
> 4. **能力雷达**：Agent 到底比裸 LLM 强在哪几维？

环境：仅 numpy / 标准库 / matplotlib。`decide()` 用规则模拟 LLM 决策，聚焦架构本身。

## 实验 1：最小可运行 Agent（ReAct 循环）

三件套齐活：**工具注册表**（说明书+函数，给 LLM 看）、**决策器**（模拟 LLM 的下一动作）、
**循环**（Thought → Action → Observation，直到 Final Answer 或步数上限）。

In [ ]:
# ===== 工具层：函数 + 给 LLM 看的说明书 =====
TOOLS = {
    "query_inventory": dict(
        desc="查询糖水店某商品库存。参数：product(str)",
        fn=lambda product: {"杨枝甘露": 42, "芒果西米露": 7, "桂花酸梅汤": 0}[product]),
    "query_order": dict(
        desc="查询订单状态。参数：order_id(str)",
        fn=lambda order_id: {"A1024": "已发货，预计明天送达", "A2048": "打包中"}[order_id]),
    "calc_discount": dict(
        desc="计算会员折扣价。参数：price(float), member(bool)",
        fn=lambda price, member: price * (0.8 if member else 1.0)),
}

# ===== 决策器：模拟 LLM 在每一步的"下一动作"（真实系统里由模型生成）=====
def decide(state):
    q = state["history"][0]["content"]
    called = {a["tool"] for a in state["actions"]}
    if "订单" in q and "query_order" not in called:
        return {"tool": "query_order", "args": {"order_id": "A1024"}}
    if "库存" in q and "query_inventory" not in called:
        return {"tool": "query_inventory", "args": {"product": "芒果西米露"}}
    if "折扣" in q and "calc_discount" not in called:
        return {"tool": "calc_discount", "args": {"price": 25.0, "member": True}}
    return {"final": state["actions"]}          # 工具用够了 → 总结回答

# ===== Agent 循环：ReAct =====
def run_agent(question, max_steps=5):
    state = {"history": [{"role": "user", "content": question}], "actions": []}
    for step in range(1, max_steps + 1):
        act = decide(state)
        if "final" in act:
            print(f"[step{step}] Final Answer: 基于 {len(act['final'])} 次工具结果回答用户 ✓")
            return act["final"]
        obs = TOOLS[act["tool"]]["fn"](**act["args"])          # 执行工具
        state["actions"].append({"tool": act["tool"], "args": act["args"], "obs": obs})
        print(f"[step{step}] Action: {act['tool']}({act['args']})")
        print(f"         Observation: {obs}")
    print("[abort] 达到步数上限，未产出答案（循环上限是防失控的关键设计）")

print("== 场景1：单工具 =="); run_agent("我的订单 A1024 到哪了？")
print("\n== 场景2：多工具组合（未预先写死的组合）==")
run_agent("芒果西米露还有库存吗？库存少的话，用会员折扣价买一杯大概多少钱？")

## 实验 2：传统规则引擎 vs Agent —— 组合爆炸

传统编程要为**每种问法**预先写规则。用户提问是工具的任意组合（2^k - 1 种非空组合），
规则表只能覆盖写过的那几条；Agent 只要每个工具单点可用 + 会规划，就能覆盖所有组合。

In [ ]:
import itertools

tool_names = ["查库存", "查订单", "算折扣", "查天气"]
combos = [c for r in range(1, len(tool_names) + 1) for c in itertools.combinations(tool_names, r)]
print(f"4 个工具 → 用户可能的组合请求 {len(combos)} 种（未含参数与语序变化）：")

RULES = {("查库存",), ("查订单",), ("查库存", "算折扣")}          # 规则表只写了 3 条
def rule_engine(query_tools): return query_tools in RULES

def agent_plan(query_tools):                                     # Agent：按需动态规划
    return all(t in tool_names for t in query_tools)

tests = [("查库存",), ("查订单", "查天气"), ("算折扣", "查天气", "查库存"), ("查库存", "算折扣")]
print(f"{'请求组合':<28}{'规则引擎':<10}{'Agent'}")
ok_rule = ok_agent = 0
for t in tests:
    r, a = rule_engine(t), agent_plan(t)
    ok_rule += r; ok_agent += a
    print(f"{'+'.join(t):<28}{('✓' if r else '✗ 未写过'):<10}{'✓ 动态规划'}")
print(f"\n覆盖率：规则引擎 {ok_rule}/{len(tests)} | Agent {ok_agent}/{len(tests)}")
print("解读：规则引擎在组合空间里必然漏；Agent 把'穷举组合'换成'组合单点能力'。")

## 实验 3：记忆组件 —— 三种上下文策略的 token 账

多轮对话中历史消息 token 只增不减。模拟 30 轮对话（每轮用户+助手+工具结果），
比较：**全量保留** / **滑动窗口 k=6** / **窗口+更早内容摘要成 1 条**。

In [ ]:
import numpy as np
rng = np.random.default_rng(11)

def simulate_dialogue(n_turns=30):
    """每轮：用户消息 + 工具结果 + 助手回答，token 数带随机性。"""
    return [int(rng.uniform(60, 140)) + int(rng.uniform(80, 200)) + int(rng.uniform(40, 90))
            for _ in range(n_turns)]

turns = simulate_dialogue()
strategies = {
    "全量保留":   lambda i: sum(turns[:i + 1]),
    "滑动窗口k=6": lambda i: sum(turns[max(0, i - 5):i + 1]),
    "窗口+摘要":   lambda i: 120 + sum(turns[max(0, i - 5):i + 1]),   # 更早的压成 120 token 摘要
}
print(f"{'策略':<12}{'第10轮':>8}{'第20轮':>8}{'第30轮':>8}{'累计token开销':>12}")
for name, f in strategies.items():
    cum = sum(f(i) for i in range(len(turns)))
    print(f"{name:<12}{f(9):>8}{f(19):>8}{f(29):>8}{cum:>12,}")
print("\n解读：全量保留 token 线性膨胀→又贵又容易超上下文；")
print("窗口+摘要是折中：近期细节保留、远期压成要点（真实 Agent 记忆系统的雏形）。")

## 实验 4：可视化 —— 能力雷达 + 记忆策略 token 曲线

In [ ]:
# matplotlib 中文字体配置（NotoSansCJK，每次画图前先跑这段）
from matplotlib import font_manager
import matplotlib.pyplot as plt

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = fontManager_font = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False

import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.4))

# 左：能力雷达图
dims = ["推理规划", "工具使用", "记忆", "自主决策", "主动性", "确定性"]
bare_llm  = [3, 1, 1, 1, 1, 2]
llm_agent = [4, 5, 4, 5, 5, 3]
angles = np.linspace(0, 2 * np.pi, len(dims), endpoint=False).tolist()
angles += angles[:1]
for vals, name, color in [(bare_llm, "裸 LLM", "#adb5bd"), (llm_agent, "LLM Agent", "#fb8500")]:
    v = vals + vals[:1]
    axes[0].plot(angles, v, "o-", color=color, label=name)
    axes[0].fill(angles, v, color=color, alpha=0.15)
axes[0].set_xticks(angles[:-1]); axes[0].set_xticklabels(dims, fontsize=9)
axes[0].set_ylim(0, 5); axes[0].set_title("裸 LLM vs LLM Agent 能力画像")
axes[0].legend(loc="lower right", fontsize=9)

# 右：记忆策略 token 曲线（实验3 的可视化）
xs = np.arange(len(turns))
for name, f, color in [("全量保留", strategies["全量保留"], "#d62828"),
                       ("滑动窗口k=6", strategies["滑动窗口k=6"], "#219ebc"),
                       ("窗口+摘要", strategies["窗口+摘要"], "#2a9d8f")]:
    axes[1].plot(xs, [f(i) for i in xs], color=color, label=name)
axes[1].set_xlabel("对话轮次"); axes[1].set_ylabel("单次请求携带 token")
axes[1].set_title("记忆策略：上下文 token 增长曲线")
axes[1].legend(fontsize=9); axes[1].grid(alpha=0.3)

plt.tight_layout(); plt.show()
print("Agent 的增益不是'模型变聪明了'，而是架构把工具/记忆/循环补到了模型短板上；")
print("代价：链路更长、更难调试 → 这是 W6-D5 框架设计要解决的问题。")

## 小结

- Agent = LLM 决策 + 工具注册表 + 记忆 + 带步数上限的循环（防失控）
- 组合爆炸让规则引擎必然漏请求；Agent 用"动态规划工具序列"覆盖长尾
- 记忆策略决定 token 成本与上下文上限；窗口+摘要是常用折中
- 下一步：D2 Function Calling、D3 ReAct、D6 完整搭建